yield 依赖解决的是:每个请求开始时创建资源 → 路由使用 → 该请求结束时释放资源

但是有些资源非常昂贵不应该每个请求都创建。
- 数据库 Engine/连接池
- Redis 客户端或连接池
- HTTP 客户端
- 机器学习模型
- Agent 使用的向量模型
- 应用级配置和缓存

更适合：应用启动时创建一次 → 多个请求共同使用 → 应用关闭时统一释放，这就是Lifespan

### Lifespan 是什么？

表示 FastAPI 应用从启动到关闭的整个生命周期
```
应用启动
   ↓
初始化共享资源
   ↓
开始接收大量请求
   ↓
应用准备关闭
   ↓
释放共享资源
```

FastAPI 推荐使用异步上下文管理器定义

In [ ]:
from contextlib import asynccontextmanager

from fastapi import FastAPI

# 需要asynccontextmanager装饰器是因为Lifespan需要支持异步上下文管理
@asynccontextmanager
async def lifespan(app: FastAPI):
    print("应用启动")

    yield # yield之前属于启动阶段，之后属于关闭阶段

    print("应用关闭")


app = FastAPI(lifespan=lifespan)

In [ ]:
# 启动时加载共享资源——以客户端为例
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.http_client = httpx.AsyncClient(
        timeout=10,
    )

    try:
        yield
    finally:
        await app.state.http_client.aclose()


app = FastAPI(lifespan=lifespan)

In [ ]:
# 路由中使用
from fastapi import Request


@app.get("/external-data")
async def external_data(request: Request):
    client = request.app.state.http_client

    response = await client.get(
        "https://example.com/api"
    )

    return response.json()

### app.state

用来保存与当前 FastAPI 应用实例关联的状态
```
app.state.http_client
app.state.redis
app.state.model
app.state.db_engine
```

### 应用级资源和请求级资源
应用级:
数据库 Engine 或连接池：应用启动时创建 -> 多个请求共享连接池 -> 应用关闭时销毁
请求级:
数据库 Session：请求开始时创建 -> 当前请求使用 -> 请求结束时关闭

### 关闭阶段与优雅关闭
应用可能因为以下原因关闭：
- 手动停止服务
- 容器收到终止信号
- 发布新版本
- Worker 重启
- 自动扩缩容
- 服务器关闭
关闭阶段应释放：
- 数据库连接池
- Redis 连接
- HTTP Client
- 后台任务
- 文件句柄
- 模型或 GPU 资源
推荐使用 try/finally：

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    resource = await create_resource()
    app.state.resource = resource

    try:
        yield
    finally:
        await resource.close()

即使应用运行期间发生问题，退出上下文时仍会尝试执行清理。

Lifespan 管理应用活多久，yield Dependency 管理请求用多久。

### 多 Worker 是最重要的陷阱

`uvicorn main:app --workers 4`代表运行4个worker进程。

每个 Worker 都有：
- 自己的 Python 解释器
- 自己的内存空间
- 自己的事件循环
- 自己的 FastAPI 应用实例
- 自己的 Lifespan
- 自己的 app.state

这些跨进程状态通常需要外部系统：
- Redis
- PostgreSQL
- 消息队列
- 专门的状态服务




## 面试
>FastAPI Lifespan 是应用级异步上下文管理器，在 yield 前执行启动初始化，在 yield 后执行关闭清理。它适合管理数据库 Engine、连接池、HTTP Client 和模型等跨请求复用的资源。请求级 Session 则应由 yield Dependency 管理。多 Worker 部署时每个 Worker 都是独立进程，会分别执行 Lifespan 并拥有独立的 app.state，所以进程内对象不能作为跨 Worker 共享状态。
